<a href="https://colab.research.google.com/github/asirabm/cctv-video-anomaly-detection/blob/main/03_normal_video_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Normal Video Feature Extraction – Part 1**

**Mount Google Drive**

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


**Create project directories**

In [3]:

import os
project_path = "/content/drive/MyDrive/UCF_Crime_Project"

save_folder = os.path.join(
    project_path, "features", "train", "Normal"
)

os.makedirs(save_folder, exist_ok=True)

print("Project directory:", project_path)
print("Feature output directory:", save_folder)
print("Output folder created:", os.path.exists(save_folder))

Project directory: /content/drive/MyDrive/UCF_Crime_Project
Feature output directory: /content/drive/MyDrive/UCF_Crime_Project/features/train/Normal
Output folder created: True


**Download the Part 1 dataset**

In [4]:

from huggingface_hub import hf_hub_download

download_folder = "/content/UCF_Crime/downloads"

os.makedirs(download_folder, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="jinmang2/ucf_crime",
    filename="Training-Normal-Videos-Part-1.zip",
    repo_type="dataset",
    local_dir=download_folder
)

print("Download completed.")
print("ZIP file:", zip_path)
print(
    "File size (GB):",
    round(os.path.getsize(zip_path) / (1024 ** 3), 2)
)

Training-Normal-Videos-Part-1.zip: reconstructing file:   0%|          |  0.00B / 38.1GB            

Training-Normal-Videos-Part-1.zip: downloading bytes:           |  0.00B            

Download completed.
ZIP file: /content/UCF_Crime/downloads/Training-Normal-Videos-Part-1.zip
File size (GB): 35.46


**Verify the ZIP contents**

In [5]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    video_files = [
        name for name in zip_ref.namelist()
        if name.lower().endswith(".mp4")
    ]

print("Total videos:", len(video_files))

print("\nFirst 10 video filenames:")
for filename in video_files[:10]:
    print(filename)

Total videos: 430

First 10 video filenames:
Training-Normal-Videos-Part-1/Normal_Videos296_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos457_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos380_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos074_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos407_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos320_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos162_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos470_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos115_x264.mp4
Training-Normal-Videos-Part-1/Normal_Videos007_x264.mp4


**Load the Pretrained ResNet50 Model**

In [8]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cpu")
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

model.fc = torch.nn.Identity()

model = model.to(device)
model.eval()
preprocess = weights.transforms()

print("ResNet50 model loaded successfully.")
print("Feature dimension: 2048")

ResNet50 model loaded successfully.
Feature dimension: 2048


**Video Frame Sampling and Feature Extraction**

In [9]:
import shutil
import tempfile

import cv2
import numpy as np
import torch

from PIL import Image
from tqdm.auto import tqdm


def extract_video_features(video_path):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError("Unable to open video")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < 1:
        cap.release()
        raise ValueError("Video contains no readable frames")

    # Select 16 frames evenly across the video
    frame_indices = np.linspace(
        0, total_frames - 1, 16
    ).astype(int)

    frames = []

    for frame_index in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(
                frame, cv2.COLOR_BGR2RGB
            )
            frames.append(Image.fromarray(frame))

    cap.release()

    if len(frames) != 16:
        raise ValueError(
            f"Expected 16 frames, but obtained {len(frames)}"
        )

    # Apply ResNet50 preprocessing
    batch = torch.stack([
        preprocess(frame) for frame in frames
    ]).to(device)

    # Extract features
    with torch.inference_mode():
        features = model(batch)

    return features.cpu().numpy().astype(np.float32)

**Extract Features from Normal Training Videos – Part 1**

In [10]:
successful = 0
skipped = 0
failed = []

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    video_files = [
        name for name in zip_ref.namelist()
        if name.lower().endswith(".mp4")
    ]

    for video_name in tqdm(video_files, desc="Processing normal videos"):

        filename = os.path.splitext(
            os.path.basename(video_name)
        )[0] + ".npy"

        feature_path = os.path.join(save_folder, filename)

        # Skip existing valid feature files
        if os.path.exists(feature_path):
            try:
                existing = np.load(feature_path, allow_pickle=False)

                if existing.shape == (16, 2048):
                    skipped += 1
                    continue
            except Exception:
                pass

        temp_video = None

        try:
            # Extract one video to temporary storage
            with tempfile.NamedTemporaryFile(
                suffix=".mp4", delete=False
            ) as temp:
                temp_video = temp.name

                with zip_ref.open(video_name) as source:
                    shutil.copyfileobj(source, temp)

            # Extract ResNet50 features
            video_features = extract_video_features(temp_video)

            # Save features to Google Drive
            temp_feature_path = feature_path + ".tmp"

            with open(temp_feature_path, "wb") as f:
                np.save(f, video_features)

            os.replace(temp_feature_path, feature_path)

            successful += 1

        except Exception as error:
            failed.append((video_name, str(error)))

        finally:
            if temp_video and os.path.exists(temp_video):
                os.remove(temp_video)

print("\nProcessing finished.")
print("New feature files saved:", successful)
print("Existing valid files skipped:", skipped)
print("Failed videos:", len(failed))

if failed:
    print("\nFailed video details:")
    for name, error in failed[:10]:
        print(name, "->", error)

Processing normal videos:   0%|          | 0/430 [00:00<?, ?it/s]


Processing finished.
New feature files saved: 0
Existing valid files skipped: 430
Failed videos: 0
